In [0]:

CREATE SCHEMA IF NOT EXISTS bronze_layer;
CREATE SCHEMA IF NOT EXISTS silver_layer;
CREATE SCHEMA IF NOT EXISTS gold_layer;

In [0]:
%python %pip install Faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 14.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
%restart_python

In [0]:
%python
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, IntegerType, FloatType
from faker import Faker
import random

# 1. Initialize Faker
fake = Faker()

@udf(returnType=StringType())
def get_name(): return fake.name()

@udf(returnType=StringType())
def get_car_number(): return fake.license_plate()

@udf(returnType=IntegerType())
def get_experience(): return random.randint(1, 15)

@udf(returnType=FloatType())
def get_rating(): return round(random.uniform(1.0, 5.0), 1)

# 2. Create the stream
streaming_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50) 
    .load()
)

# 3. Transform the stream
drivers_stream = (streaming_df
    .withColumn("id", col("value")) 
    .withColumn("name", get_name())
    .withColumn("car_number", get_car_number())
    .withColumn("experience", get_experience())
    .withColumn("rating", get_rating())
    .drop("value", "timestamp") 
)

# ---------------------------------------------------------
# THE FIX: Create a Unity Catalog Volume for Checkpoints
# ---------------------------------------------------------
# Get the current default catalog (usually 'workspace' in Free Edition)
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]

# Create the volume inside our bronze_layer schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS {current_catalog}.bronze_layer.checkpoints")

# Define the new, secure path
checkpoint_path = f"/Volumes/{current_catalog}/bronze_layer/checkpoints/bronze_drivers"
print(f"Secure Volume created. Using checkpoint path: {checkpoint_path}")
# ---------------------------------------------------------

# 4. Start the stream with Serverless-compatible Trigger
query = (drivers_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) 
    .table("bronze_layer.drivers_raw")
)

query.awaitTermination() 
print("Batch complete! Data successfully streamed to Bronze.")

Secure Volume created. Using checkpoint path: /Volumes/workspace/bronze_layer/checkpoints/bronze_drivers
Batch complete! Data successfully streamed to Bronze.


In [0]:
SELECT * FROM bronze_layer.drivers_raw limit 10;

id,name,car_number,experience,rating,phone
903637,Shannon Gordon,86-D411,9,1.7,718.264.1121x226
903645,Dennis Jennings,HDE 265,12,1.3,001-959-853-8689x179
903653,Christopher Taylor,595 3AW,6,2.6,(402)517-7644x0970
903661,Jennifer Diaz,ZMQ-917,14,2.1,+1-559-984-5672x27010
903669,Matthew Berry,866 0OV,6,4.0,(663)966-4782x634
903677,Stephen Cruz,21-77644,10,3.3,920-317-5489x660
903685,Terri Jones,70TW235,9,4.6,4739908802
903693,Laura Stanley,56Q I72,10,1.8,791.327.4250
903701,Janice Wise,KYV 701,5,4.2,+1-914-423-2980x0013
903709,Jillian Powell,8375,8,1.3,+1-832-571-0057x598


In [0]:
%python
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, IntegerType, FloatType
from faker import Faker
import random

fake = Faker()

# Keep our previous functions
@udf(returnType=StringType())
def get_name(): return fake.name()

@udf(returnType=StringType())
def get_car_number(): return fake.license_plate()

@udf(returnType=IntegerType())
def get_experience(): return random.randint(1, 15)

@udf(returnType=FloatType())
def get_rating(): return round(random.uniform(1.0, 5.0), 1)

# NEW FUNCTION: Generate a fake phone number
@udf(returnType=StringType())
def get_phone(): return fake.phone_number()

# Create the stream again
streaming_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50) 
    .load()
)

# Transform the stream AND include the new 'phone' column
drivers_stream_evolved = (streaming_df
    .withColumn("id", col("value")) 
    .withColumn("name", get_name())
    .withColumn("car_number", get_car_number())
    .withColumn("experience", get_experience())
    .withColumn("rating", get_rating())
    .withColumn("phone", get_phone()) # <-- ADDING THE NEW FIELD
    .drop("value", "timestamp") 
)

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
checkpoint_path = f"/Volumes/{current_catalog}/bronze_layer/checkpoints/bronze_drivers"

# Start the stream with Schema Evolution turned ON (.option("mergeSchema", "true"))
query = (drivers_stream_evolved.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true") # <-- CRITICAL FOR SCHEMA EVOLUTION!
    .trigger(availableNow=True) 
    .table("bronze_layer.drivers_raw")
)

query.awaitTermination() 
print("Evolved batch complete! New 'phone' column added automatically.")

Evolved batch complete! New 'phone' column added automatically.


In [0]:
SELECT * FROM bronze_layer.drivers_raw VERSION AS OF 1 LIMIT 10;

id,name,car_number,experience,rating
3,Bonnie Cox,ZER 546,13,2.1
11,Theresa Miller,875-RDE,10,3.1
19,Justin Nolan,WOG 388,2,3.5
27,Mark Benson MD,LXT 889,6,4.6
5,Bonnie Cox,ZER 546,2,4.4
13,Theresa Miller,875-RDE,3,1.6
21,Justin Nolan,WOG 388,6,1.4
6,Bonnie Cox,ZER 546,10,4.4
14,Theresa Miller,875-RDE,8,3.7
22,Justin Nolan,WOG 388,14,4.4


In [0]:
RESTORE TABLE bronze_layer.drivers_raw TO VERSION AS OF 1;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
14074,8,9,8,10166807,14074


In [0]:
SELECT * FROM bronze_layer.drivers_raw LIMIT 10;

id,name,car_number,experience,rating
0,Bonnie Cox,ZER 546,11,1.9
8,Theresa Miller,875-RDE,11,4.9
16,Justin Nolan,WOG 388,7,3.7
24,Mark Benson MD,LXT 889,10,3.1
1,Bonnie Cox,ZER 546,4,4.6
9,Theresa Miller,875-RDE,3,2.1
17,Justin Nolan,WOG 388,8,3.5
25,Mark Benson MD,LXT 889,5,3.8
2,Bonnie Cox,ZER 546,12,4.8
10,Theresa Miller,875-RDE,6,4.6


In [0]:
-- 1. Create the cleaned Silver table from our Bronze data
CREATE TABLE IF NOT EXISTS silver_layer.drivers_clean AS
SELECT 
    CAST(id AS INT),
    name,
    car_number,
    CAST(experience AS INT),
    CAST(rating AS FLOAT)
FROM bronze_layer.drivers_raw;

-- 2. Apply Z-Ordering optimization on the experience column
OPTIMIZE silver_layer.drivers_clean ZORDER BY (experience);

-- 3. Create a static vehicle lookup table (Table #3 for your requirement)
CREATE TABLE IF NOT EXISTS silver_layer.vehicles (
    car_number STRING,
    vehicle_type STRING
);

-- Insert some dummy data so it isn't empty
INSERT INTO silver_layer.vehicles VALUES 
('ABC-123', 'Sedan'), 
('XYZ-987', 'SUV');

num_affected_rows,num_inserted_rows
2,2


In [0]:
-- 4. Create an active drivers view with Liquid Clustering (Table #4)
CREATE TABLE IF NOT EXISTS gold_layer.drivers_active
CLUSTER BY (id) 
AS 
SELECT 
    d.id,
    d.name,
    d.car_number,
    COALESCE(v.vehicle_type, 'Unknown') AS vehicle_type
FROM silver_layer.drivers_clean d
LEFT JOIN silver_layer.vehicles v ON d.car_number = v.car_number;

-- 5. Create an aggregated statistics table (Table #5)
CREATE TABLE IF NOT EXISTS gold_layer.driver_stats AS
SELECT 
    experience,
    ROUND(AVG(rating), 2) AS avg_rating,
    COUNT(id) AS total_drivers
FROM silver_layer.drivers_clean
GROUP BY experience;

num_affected_rows,num_inserted_rows


In [0]:
-- NOTE FOR THE Mentor
-- 1. The Databricks Free Edition restricts creating account-level groups via SQL, so I used my actual user email.
-- 2. Unity Catalog does not support the legacy `DENY` command. To explicitly restrict WRITE (MODIFY) 
--    permissions in a UC environment, the `REVOKE` command must be used.

-- Grant READ permissions on the schema and tables
GRANT SELECT ON SCHEMA gold_layer TO `temur.botchoridze@innowise.com`;
GRANT SELECT ON TABLE gold_layer.drivers_active TO `temur.botchoridze@innowise.com`;
GRANT SELECT ON TABLE gold_layer.driver_stats TO `temur.botchoridze@innowise.com`;

-- Explicitly restrict WRITE (MODIFY) permissions using REVOKE
REVOKE MODIFY ON SCHEMA gold_layer FROM `temur.botchoridze@innowise.com`;
REVOKE MODIFY ON TABLE gold_layer.drivers_active FROM `temur.botchoridze@innowise.com`;
REVOKE MODIFY ON TABLE gold_layer.driver_stats FROM `temur.botchoridze@innowise.com`;